In [ ]:
# Clasificación de enfermedades del limon atraves de un modelo de aprendisaje Utilizando y Tensorflow keras – CNN

# Integrantes: Caleb Seña Melo y Santiago Maya Ospina
# link del dataset: https://www.kaggle.com/datasets/chayanmondalabir/citrus-leaf-pathology-multi-class-image-dataset


In [ ]:

from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import zipfile
import os
import shutil


dataset_name = "archive.zip"
zip_path = "/content/drive/MyDrive/dataset/" + dataset_name
extract_path = "/content/dataset_limpio"


if os.path.exists(extract_path):
    shutil.rmtree(extract_path)
    
os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)


print("¡Dataset descomprimido correctamente!")
print("Contenido:")
print(os.listdir(extract_path))


In [ ]:
print("Contenido de /content/dataset_limpio:")
print(os.listdir("/content/dataset_limpio"))

In [ ]:
BASE_PATH = "/content/dataset_limpio"

DATASET_PATH = os.path.join(
    BASE_PATH,
    "CitrusLeafPathology-14-Aug",
    "CitrusLeafPathology-14-Aug"
)

TRAIN_PATH = os.path.join(DATASET_PATH, "train")
VAL_PATH = os.path.join(DATASET_PATH, "val")
TEST_PATH = os.path.join(DATASET_PATH, "test")

METADATA_PATH = os.path.join(
    BASE_PATH,
    "CLP-14-Aug_metadata.csv"
)

print("TRAIN:", TRAIN_PATH)
print("VAL:", VAL_PATH)
print("TEST:", TEST_PATH)
print("METADATA:", METADATA_PATH)

In [ ]:
import tensorflow as tf
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import os

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import (
    Conv2D,
    MaxPooling2D,
    GlobalAveragePooling2D,
    Dense,
    Dropout,
    Input,
    BatchNormalization,
    Activation
)

from tensorflow.keras.callbacks import (
    ReduceLROnPlateau,
    EarlyStopping,
    ModelCheckpoint
)

from tensorflow.keras.regularizers import l2
from tensorflow.keras.optimizers import Adam

from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import confusion_matrix, classification_report

import seaborn as sns

In [ ]:
tf.keras.utils.set_random_seed(42)

print("Semilla establecida.")

In [ ]:
df = pd.read_csv(METADATA_PATH)

print("Columnas:")
print(df.columns)

print("\nPrimeras filas:")
display(df.head())

In [ ]:
conteo = df["class_name"].value_counts()

print("Cantidad de imágenes por clase:")
print(conteo)

print("\nTotal de imágenes:", len(df))
print("Mínimo:", conteo.min())
print("Máximo:", conteo.max())
print("Promedio:", conteo.mean())

In [ ]:
print("Clases del dataset:")

for i, clase in enumerate(sorted(df["class_name"].unique())):
    print(f"{i}: {clase}")

In [ ]:
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    width_shift_range=0.05,
    height_shift_range=0.05,
    zoom_range=0.1,
    horizontal_flip=True,
    vertical_flip=True,
    fill_mode="nearest"
)

In [ ]:
val_test_datagen = ImageDataGenerator(
    rescale=1./255
)

In [ ]:
train_generator = train_datagen.flow_from_directory(
    TRAIN_PATH,

    target_size=(224, 224),

    batch_size=32,

    class_mode="categorical",

    shuffle=True,

    seed=42
)
val_generator = val_test_datagen.flow_from_directory(
    VAL_PATH,

    target_size=(224, 224),

    batch_size=32,

    class_mode="categorical",

    shuffle=False
)
test_generator = val_test_datagen.flow_from_directory(
    TEST_PATH,

    target_size=(224, 224),

    batch_size=32,

    class_mode="categorical",

    shuffle=False
)

In [ ]:
print("Imágenes de entrenamiento:", train_generator.samples)
print("Imágenes de validación:", val_generator.samples)
print("Imágenes de prueba:", test_generator.samples)

print("\nNúmero de clases:", train_generator.num_classes)

In [ ]:
print("Clases TRAIN:")
print(train_generator.class_indices)

print("\nClases VALIDACIÓN:")
print(val_generator.class_indices)

print("\nClases TEST:")
print(test_generator.class_indices)

In [ ]:
classes = np.unique(train_generator.classes)

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=train_generator.classes
)

class_weight_dict = dict(
    zip(classes, class_weights)
)

print("Pesos de las clases:")

for clase_id, peso in class_weight_dict.items():
    print(
        f"{clase_id} - "
        f"{list(train_generator.class_indices.keys())[list(train_generator.class_indices.values()).index(clase_id)]}: "
        f"{peso:.4f}"
    )

In [ ]:
model = Sequential([
    
    Input(shape=(224, 224,3)),
    Conv2D(64, (3,3), padding="same", kernel_regularizer=l2(0.0001)),
    BatchNormalization(),
    Activation("relu"),
    MaxPooling2D(pool_size=(2,2)),

    Conv2D(128, (3,3), padding="same", kernel_regularizer=l2(0.0001)),
    BatchNormalization(),
    Activation("relu"),
    MaxPooling2D(pool_size=(2,2)),

    Conv2D(256, (3,3), padding="same", kernel_regularizer=l2(0.0001)),
    BatchNormalization(),
    Activation("relu"),
    MaxPooling2D(pool_size=(2,2)),

    Conv2D(512, (3,3), padding="same", kernel_regularizer=l2(0.0001)),
    BatchNormalization(),
    Activation("relu"),
    MaxPooling2D(pool_size=(2,2)),
    Dropout(0.30),
    GlobalAveragePooling2D(),
    Dense(512,kernel_regularizer=l2(0.0001)),
    Activation("relu"),
    Dropout(0.5),
    Dense(train_generator.num_classes, activation="softmax")
])
model.compile(optimizer=Adam(learning_rate=0.0001),
              loss="categorical_crossentropy",
              metrics=["accuracy"])
model.summary()

In [ ]:
checkpoint = [
    ModelCheckpoint(
    filepath="mejor_modelo.keras",
    monitor="val_accuracy",
    save_best_only=True,
    mode="max",
    verbose=1
),
EarlyStopping(
    monitor="val_accuracy",
    patience=10,
    restore_best_weights=True,
    verbose=1
),
ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.2,
    patience=3,
    min_lr=1e-6,
    verbose=1
)
]

In [ ]:
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=40,
   callbacks=[checkpoint]
)

In [ ]:
best_model = load_model("mejor_modelo.keras")
loss, acc = best_model.evaluate(test_generator)
print(f"Accuracy en test: {acc:.4f}")
print(f"Loss en test: {loss:.4f}")

In [ ]:
plt.figure(figsize=(10, 5))

plt.plot(
    history.history["accuracy"],
    label="Entrenamiento"
)

plt.plot(
    history.history["val_accuracy"],
    label="Validación"
)

plt.title("Accuracy durante el entrenamiento")

plt.xlabel("Época")
plt.ylabel("Accuracy")

plt.legend()
plt.grid(True)

plt.show()

In [ ]:
plt.figure(figsize=(10, 5))

plt.plot(
    history.history["loss"],
    label="Entrenamiento"
)

plt.plot(
    history.history["val_loss"],
    label="Validación"
)

plt.title("Loss durante el entrenamiento")

plt.xlabel("Época")
plt.ylabel("Loss")

plt.legend()
plt.grid(True)

plt.show()

In [ ]:
best_model = load_model("mejor_modelo.keras")

print("Mejor modelo cargado correctamente.")

In [ ]:
loss, acc = best_model.evaluate(
    test_generator,
    verbose=1
)

print(f"\nAccuracy en test: {acc:.4f}")
print(f"Accuracy en test: {acc * 100:.2f}%")

print(f"Loss en test: {loss:.4f}")

In [ ]:
test_generator.reset()

y_pred = best_model.predict(
    test_generator,
    verbose=1
)

y_pred_classes = np.argmax(
    y_pred,
    axis=1
)

y_true = test_generator.classes

In [ ]:
class_names = list(test_generator.class_indices.keys())

print(class_names)

In [ ]:
cm = confusion_matrix(
    y_true,
    y_pred_classes
)

plt.figure(figsize=(12, 10))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names
)

plt.title("Matriz de confusión")

plt.xlabel("Predicción")
plt.ylabel("Clase real")

plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)

plt.tight_layout()

plt.show()

In [ ]:
print(
    classification_report(
        y_true,
        y_pred_classes,
        target_names=class_names,
        digits=4
    )
)

In [ ]:
cm = confusion_matrix(y_true, y_pred_classes)

accuracy_por_clase = []

for i, clase in enumerate(class_names):

    total_clase = cm[i].sum()

    correctas = cm[i, i]

    accuracy_clase = correctas / total_clase

    accuracy_por_clase.append(
        accuracy_clase
    )

df_accuracy_clases = pd.DataFrame({
    "Clase": class_names,
    "Accuracy": accuracy_por_clase
})

df_accuracy_clases["Accuracy (%)"] = (
    df_accuracy_clases["Accuracy"] * 100
)

df_accuracy_clases = df_accuracy_clases.sort_values(
    "Accuracy"
)

display(df_accuracy_clases)

In [ ]:
plt.figure(figsize=(12, 6))

plt.bar(
    df_accuracy_clases["Clase"],
    df_accuracy_clases["Accuracy (%)"]
)

plt.title("Accuracy por clase")

plt.xlabel("Clase")
plt.ylabel("Accuracy (%)")

plt.xticks(
    rotation=45,
    ha="right"
)

plt.grid(axis="y")

plt.tight_layout()

plt.show()